In [1]:
import openvino as ov
import sys
import platform

print("Python version:", sys.version)
print("OS:", platform.platform())
print("OpenVINO version:", ov.get_version())

Python version: 3.9.10 (tags/v3.9.10:f2f3f53, Jan 17 2022, 15:14:21) [MSC v.1929 64 bit (AMD64)]
OS: Windows-10-10.0.26200-SP0
OpenVINO version: 2023.1.0-12185-9e6b00e51cd-releases/2023/1


In [2]:
from pathlib import Path

# Path to ONNX model (from Notebook 1)
ONNX_MODEL_PATH = Path("ssdlite_mobilenet_v3_1.onnx")

# Output directory for OpenVINO IR
IR_OUTPUT_DIR = Path("openvino_ir1")
IR_OUTPUT_DIR.mkdir(exist_ok=True)

print("ONNX model path:", ONNX_MODEL_PATH)
print("IR output directory:", IR_OUTPUT_DIR)


ONNX model path: ssdlite_mobilenet_v3_1.onnx
IR output directory: openvino_ir1


In [3]:
import openvino as ov
import cv2
import numpy as np
import time


In [4]:
core = ov.Core()

model_path = "openvino_ir1/ssdlite_mobilenet_v3.xml"  # FP16 model
compiled_model = core.compile_model(model_path, "CPU")

input_layer = compiled_model.input(0)
output_layers = compiled_model.outputs

print("OpenVINO model loaded for real-time inference")


OpenVINO model loaded for real-time inference


In [5]:
coco_classes = [
    "__background__", "person", "bicycle", "car", "motorcycle",
    "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "bench", "bird", "cat", "dog"
]


In [6]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "Webcam not accessible"

input_height, input_width = 320, 320
confidence_threshold = 0.5


In [7]:
prev_time = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Preprocess
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(frame_rgb, (input_width, input_height))
    input_image = resized.astype(np.float32) / 255.0
    input_image = np.transpose(input_image, (2, 0, 1))
    input_image = np.expand_dims(input_image, axis=0)

    # Inference
    start_time = time.time()
    results = compiled_model([input_image])
    end_time = time.time()

    latency_ms = (end_time - start_time) * 1000
    fps = 1000 / latency_ms

    boxes = results[compiled_model.output(0)]
    labels = results[compiled_model.output(1)]
    scores = results[compiled_model.output(2)]

    h, w, _ = frame.shape

    for i in range(len(scores)):
        if scores[i] > confidence_threshold:
            box = boxes[i]
            label_id = int(labels[i])

            x1 = int(box[0] * w)
            y1 = int(box[1] * h)
            x2 = int(box[2] * w)
            y2 = int(box[3] * h)

            class_name = coco_classes[label_id] if label_id < len(coco_classes) else "object"

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                frame,
                f"{class_name} {scores[i]:.2f}",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    # Overlay performance info
    cv2.putText(
        frame,
        f"Latency: {latency_ms:.1f} ms | FPS: {fps:.1f}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2
    )

    cv2.imshow("OpenVINO Real-Time Object Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


In [8]:
cap.release()
cv2.destroyAllWindows()
